In [3]:
import boto3
import json
import os
import pandas as pd
import numpy as np
import pprint

In [ ]:
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = "="

In [57]:
# testing model
client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-2" # If you've configured a default region, you can omit this line
)

# Define the model and message
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
messages = [{"role": "user", "content": [{"text": "Hello"}]}]
response = client.converse(
    modelId=model_id,
    messages=messages,
)
#message_text = response['output']['message']['content'][0]['text']
pprint.pprint(response)

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '344',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 06 Mar 2026 02:20:52 GMT',
                                      'x-amzn-requestid': '715353da-9771-4ab0-bc1d-c1be1ee41c61'},
                      'HTTPStatusCode': 200,
                      'RequestId': '715353da-9771-4ab0-bc1d-c1be1ee41c61',
                      'RetryAttempts': 0},
 'metrics': {'latencyMs': 1830},
 'output': {'message': {'content': [{'text': 'Hello! How can I help you '
                                             'today?'}],
                        'role': 'assistant'}},
 'stopReason': 'end_turn',
 'usage': {'cacheReadInputTokens': 0,
           'cacheWriteInputTokens': 0,
           'inputTokens': 8,
           'outputTokens': 12,
           'totalTokens': 20}}


In [5]:
file = '2018-09-13--06.02.24.events.csv'
sim_data = pd.read_csv(file, header=4)

sim_data["Info"] = sim_data["Info"].astype(str)
rows_to_drop = [31330, 4816]
sim_data = sim_data.drop(index=rows_to_drop)

# group by agent
agent_histories = {
    agent_id: group.sort_values('Time')
    for agent_id, group in sim_data.groupby('Agent')
}

C:\Users\yysma\AppData\Local\Temp\ipykernel_28936\2211920136.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  sim_data = pd.read_csv(file, header=4)


In [7]:
unique_zips = sim_data['Zip'].unique()

In [9]:
high_risk_non_hr = sim_data[
    (sim_data['Age'] >= 18) &
    (sim_data['Syringe_source'] == 'nonHR') & 
    (sim_data['Fraction_recept_sharing'] > 0) &
    ((sim_data['Drug_in_degree'] > 0) & (sim_data['Drug_out_degree'] > 0))
].copy()

#get the list of unique agents in the nonHR group
unique_non_hr_ids = high_risk_non_hr['Agent'].unique()

#shuffle the IDs once
np.random.seed(42)
shuffled_ids = np.random.permutation(unique_non_hr_ids)

#create batches of 10 unique IDs
batch_size = 10
id_batches = [shuffled_ids[i:i + batch_size] for i in range(0, len(shuffled_ids), batch_size)]

id_batches[0]

array([1332273946,  425435063,  605906258, 1312810575,  168619843,
       1984135393, 1626257718,   96703808, 1511373692,  187406409])

In [10]:
#checking
print(sim_data['Daily_injection_intensity'].max())
print(len(id_batches))
print(len(unique_non_hr_ids))
is_unique = len(shuffled_ids) == len(set(shuffled_ids))
print(len(set(shuffled_ids)))

20.0
453
4525
4525


In [61]:
#HR prompt 1
prompt_list = []
# converting the 'activated' part of an agent's dataframe history into a profile prompt

for agent_id in id_batches[0][:1]:
    agent_df = agent_histories[agent_id]
    row = agent_df[agent_df['Event'] == 'activated'].iloc[0].copy()

    #source_map = "used harm reduction services" if row['Syringe_source'] == "HR" else "not used harm reduction services"
    race_map = {
    "NHBlack": "Black",
    "NHWhite": "White",
    "Hispanic": "Hispanic",
    "Other": "a race other than Black, White, or Hispanic"
}
    hcv_map = {
    'susceptible': "Susceptible to heptatitus C virus",
    'exposed': "Exposed to hepatitis C virus",
    'infectiousacute': "Infected with hepatitis C virus",
    'chronic': "Chronic hepatitis C virus",
    'recovered': "Recovered from hepatitis C virus"
}
    cleaned_race = race_map.get(row['Race'])
    cleaned_hcv = hcv_map.get(row['HCV'])

    # building the API prompt 
    api_text = (
        #"Background Context (Chicago, 2026):\n"
        #"The drug market is dominated by synthetic opioids. Fentanyl is involved in over 82% of overdose deaths.\n"
        #"Drugs sold as heroin or cocaine often contain multiple substances (polysubstance mixtures), including synthetic fentanyl, benzodiazepines, xylazine (“tranq”), and medetomidine. Many users are unaware their drugs are contaminated.\n"
        #"Austin, East and West Garfield Park, Humboldt Park, and North Lawndale are neighborhoods where overdoses are most frequently recorded. These ZIP codes are commonly targeted for harm-reduction outreach and research sampling.\n"
        #"Among people using city harm-reduction vending machines, around 38% report being unhoused, living with someone else, or living in shelters.\n"
        #"Harm reduction is available through on-site and mobile SSPs, vending machines, SOR-ICS, and Narcan Newsstands, which provide free, anonymous naloxone and supplies. Hotlines like MAR NOW provide free medications for opioid use disorder. Participating pharmacies generally require payment or insurance.\n"
        #"While overall overdose deaths have declined, the street-level environment remains unpredictable due to polysubstances, stigma, and inconsistent access to harm reduction.\n"
        "Individual Background:\n"
        "The following information describes your typical behavior during the past 6 months BEFORE you started using a syringe service program (SSP).\n"
        f"Age: {round(row['Age'])} years old\n"
        f"Gender: {row['Gender']}\n"
        f"Race: {cleaned_race}\n"
        f"Neighborhood (ZIP code): {row['Zip']}\n"
        f"Health status: {cleaned_hcv}\n"
        f"Injection history: You began injecting drugs at {round(row['Age_Started'])} years old and had never used harm reduction services before enrolling in a SSP.\n"
        f"Injection frequency: Around {round(row['Daily_injection_intensity']*7)} injections per week. About {int(row['Fraction_recept_sharing']*100)} percent of your injections involved using a syringe that someone else had already used.\n"
        f"Social network: {round(row['Drug_in_degree'])} people give you used syringes, {round(row['Drug_out_degree'])} people receive your used syringes, and about {round(row['HCV_friend_preval'])} of your friends have hepatitis C.\n"        

        "Intervention:\n"
        "You began using a SSP one month ago. You obtain sterile syringes and harm reduction supplies from a SSP when you are able to access it.\n"

        "Task:\n"
        "Complete this anonymous survey based ONLY on your behavior during the past 7 days (after one month of SSP use).\n"

        "1. How many times did you inject drugs?\n"
        "2. Of the injections you reported, how many times did you use a needle that had already been used by someone else?\n"
        "3. Of the injections you reported, how many times were the drugs measured or mixed in someone else's syringe before being transferred to yours (backloading)?\n"
        "4. Of the injections you reported, how many times did you use a sterile syringe obtained from a SSP?\n"
        "5. What was the main drug you injected?\n"
        "6. Of the injections you reported, how many involved intentionally using a second drug together with your main drug?\n"
        "7. Of the injections you reported, how many times did you suspect another drug was present in the drug you used?\n"
        "8. How many different people provided you with used needles?\n"
        "9. How many different people did you give your used needles to?\n"
        "10. Of the injections you reported, how many times did you share a cooker, cotton, or water with someone else? Count each injection only once, even if you shared more than one item.\n"
        "11. Of the injections you reported, how many times did you safely dispose of the used syringe immediately after use?\n"
        "12. Are you currently homeless?\n"
        "13. Do you currently have access to drug test strips?\n"
        "14. Of the injections you reported, how many times did you use a drug test strip before injecting?\n"
        "15. Out of the times you used a drug test strip, how many times did a positive result for an unexpected substance lead you to not inject?\n"
        "16. How many days did you carry naloxone?\n"

        "Output Format:\n"
        "“Return your answer as a single-line JSON object with the following keys. Do not include line breaks, extra formatting, or any text outside the JSON. Unless indicated, all vallues must be integers equal to or less than the number of injections reported.\n"
        "injections (0-150)\n"
        "recept_sharing\n"
        "backloading\n"
        "sterile_ssp\n"
        "main_drug (1=heroin, 2=fentanyl, 3=cocaine, 4=methamphetamine, 5=xylazine/medetomidine, 6=other)\n"
        "drug2_aware\n"
        "drug2_suspect\n"
        "in_degree (≤ recept_sharing)\n"
        "out_degree\n"
        "equipment\n"
        "safe_disposal\n"
        "homeless (0=no, 1=yes)\n"
        "test_strip_access (0=no, 1=yes)\n"
        "test_strip_usage\n"
        "positive_test (≤ test_strips)\n"
        "naloxone_days (0-7)"

    )

    prompt_list.append(api_text)  

prompt_list
print(prompt_list[0])

Individual Background:
The following information describes your typical behavior during the past 6 months BEFORE you started using a syringe service program (SSP).
Age: 30 years old
Gender: Female
Race: White
Neighborhood (ZIP code): 60622
Health status: Chronic hepatitis C virus
Injection history: You began injecting drugs at 20 years old and had never used harm reduction services before enrolling in a SSP.
Injection frequency: Around 84 injections per week. About 48 percent of your injections involved using a syringe that someone else had already used.
Social network: 1 people give you used syringes, 2 people receive your used syringes, and about 0 of your friends have hepatitis C.
Intervention:
You began using a SSP one month ago. You obtain sterile syringes and harm reduction supplies from a SSP when you are able to access it.
Task:
Complete this anonymous survey based ONLY on your behavior during the past 7 days (after one month of SSP use).
1. How many times did you inject drugs?

for index, api_text in enumerate(prompt_list): 
    current_agent_id = id_batches[0][index]
    print(current_agent_id)
    print(type(current_agent_id.item()))

In [66]:
client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

model_id = "us.amazon.nova-2-lite-v1:0"
#"meta.llama3-70b-instruct-v1:0" 
#"us.anthropic.claude-sonnet-4-5-20250929-v1:0"
#us.amazon.nova-2-lite-v1:0"
#""

responses_list = []

for index, api_text in enumerate(prompt_list): 
    current_agent_id = id_batches[0][index]

    messages = [{
        "role": "user",
        "content": [{"text": api_text}]
    }]

    response = client.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig={
            "temperature": 0.8,
    }
    )

    # get the raw string from bedrock
    raw_message_text = response['output']['message']['content'][0]['text']

    try:
        # clean up the string and convert it to a real python dict
        structured_response = json.loads(raw_message_text.strip())
    except json.JSONDecodeError:
        structured_response = raw_message_text
    
    entry = {
    "agent": current_agent_id.item(),
    "background": "no",
    "model": "nova2-lite", #claude-sonnet4.5
    "response": structured_response
    }

    #append immediately to file
    with open("agent_responses.jsonl", "a") as f:  
        f.write(json.dumps(entry) + "\n")


In [67]:
input_file = "agent_responses.jsonl"
output_file = "agent_responses_clean.jsonl"

with open(input_file, "r") as f_in, open(output_file, "w") as f_out:
    for line in f_in:
        entry = json.loads(line)
        
        response_raw = entry["response"]
        if isinstance(response_raw, str):
            #remove slahes
            cleaned = response_raw.replace("```json", "").replace("```", "").strip()
            
            try:
                structured_response = json.loads(cleaned)
            except json.JSONDecodeError:
                structured_response = response_raw
            
            entry["response"] = structured_response
        
        f_out.write(json.dumps(entry) + "\n")